(Suite) Après l'EDA et la transformation des données, ici nous préparons les données pour le Machine Learning

### Etape 4  Préparation des données pour ML

Encode les variables catégorielles

Les modèles ML (sauf les arbres) ne comprennent pas les catégories textuelles.

Pour les variables binaires, on peut utiliser 0/1.

Pour les variables multi-classes, on fait un one-hot encoding.

In [42]:
# -----------------------
# Préparation finale des données (train/test/result)
# -----------------------
import os
import json
import datetime
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
import joblib
from pathlib import Path



In [46]:
# --- paramètres reproducibles ---
RANDOM_STATE = 42
TEST_SIZE = 0.20
OUT_DIR = "../data/models"             # dossiers de sortie
ARTIFACTS_DIR = "../data/models/artifacts"  # préprocesseur, metadata, etc.

os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(ARTIFACTS_DIR, exist_ok=True)

df_original = pd.read_parquet("../../data/processed/telco_customer_churn_processed.parquet")
df = df_original.copy()

In [31]:
df.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [32]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 20 columns):
 #   Column            Non-Null Count  Dtype   
---  ------            --------------  -----   
 0   gender            7043 non-null   category
 1   SeniorCitizen     7043 non-null   int64   
 2   Partner           7043 non-null   category
 3   Dependents        7043 non-null   category
 4   tenure            7043 non-null   int64   
 5   PhoneService      7043 non-null   category
 6   MultipleLines     7043 non-null   category
 7   InternetService   7043 non-null   category
 8   OnlineSecurity    7043 non-null   category
 9   OnlineBackup      7043 non-null   category
 10  DeviceProtection  7043 non-null   category
 11  TechSupport       7043 non-null   category
 12  StreamingTV       7043 non-null   category
 13  StreamingMovies   7043 non-null   category
 14  Contract          7043 non-null   category
 15  PaperlessBilling  7043 non-null   category
 16  PaymentMethod     7043 n

In [33]:
# --- 1) target & simple checks ---
TARGET = "Churn"
if TARGET not in df.columns:
    raise ValueError(f"{TARGET} absent du dataframe")

# Si Churn est 'Yes'/'No' (category), on le mappe en 0/1
if df[TARGET].dtype.name in ("category","object"):
    df[TARGET] = df[TARGET].map({'Yes':1, 'No':0}).astype(int)


In [34]:
# --- 2) Features lists (robuste) ---
# On choisit numeric et categorical en fonction des types actuels du dataframe
numeric_features = df.select_dtypes(include=[np.number]).columns.tolist()
# retirer la cible
numeric_features = [c for c in numeric_features if c != TARGET]

categorical_features = df.select_dtypes(include=['category','object']).columns.tolist()
# certains 'object' peuvent être des IDs -> s'assurer qu'on n'inclut pas customerID si présent
if 'customerID' in categorical_features:
    categorical_features.remove('customerID')

# Afficher listes pour vérification (optionnel)
print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)

Numeric features: ['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges']
Categorical features: ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']


In [35]:
# --- 3) FEATURE ENGINEERIN ---
# On crée AverageChargesPerMonth = TotalCharges / tenure (protéger division par 0)
if set(['TotalCharges','tenure']).issubset(df.columns):
    df['AvgChargesPerMonth'] = df.apply(
        lambda row: row['TotalCharges'] / row['tenure'] if row['tenure'] > 0 else row['MonthlyCharges'], axis=1
    )
    # ajouter à numeric_features
    if 'AvgChargesPerMonth' not in numeric_features:
        numeric_features.append('AvgChargesPerMonth')

In [36]:
# --- 4) Split train/test (stratified) ---
X = df.drop(columns=[TARGET])
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)

print("Train shape:", X_train.shape, "Test shape:", X_test.shape)
print("Train churn %:\n", y_train.value_counts(normalize=True))
print("Test churn %:\n", y_test.value_counts(normalize=True))

Train shape: (5634, 20) Test shape: (1409, 20)
Train churn %:
 Churn
0    0.734647
1    0.265353
Name: proportion, dtype: float64
Test churn %:
 Churn
0    0.734564
1    0.265436
Name: proportion, dtype: float64


In [37]:
# --- 5) Preprocessing pipeline (fit ONLY on train) ---
# numeric transformer: imputer + scaler
# numeric transformer: imputer + scaler
numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# categorical transformer: imputer (rare) + one-hot (ignore unseen)
categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value='MISSING')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))  # <= correction ici
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ],
    remainder='drop'
)

# Fit sur X_train et transformer
preprocessor.fit(X_train)

X_train_proc = preprocessor.transform(X_train)
X_test_proc  = preprocessor.transform(X_test)

In [38]:
# --- 6) Recover feature names to build DataFrame back (pro) ---
# Note: get_feature_names_out requires sklearn >= 1.0
num_features_out = numeric_features
cat_features_out = []
if len(categorical_features) > 0:
    cat_ohe = preprocessor.named_transformers_['cat'].named_steps['onehot']
    cat_features_out = list(cat_ohe.get_feature_names_out(categorical_features))
feature_names = list(num_features_out) + cat_features_out

# Créer DataFrames avec index d'origine (pratique pour mapping)
X_train_df = pd.DataFrame(X_train_proc, columns=feature_names, index=X_train.index)
X_test_df  = pd.DataFrame(X_test_proc, columns=feature_names, index=X_test.index)

# Joindre la target
train_df = X_train_df.copy()
train_df[TARGET] = y_train

test_df = X_test_df.copy()
test_df[TARGET] = y_test

In [39]:
# --- 7) Créer template results (vide) pour stocker prédictions futures ---
results_df = X_test_df.copy()
results_df['true'] = y_test
results_df['predicted'] = np.nan
results_df['predicted_proba'] = np.nan


In [48]:
# --- 8) Sauvegarder fichiers (parquet si possible) ---
# Parquet est recommandé (plus compact, types préservés). Si pyarrow absent -> fallback CSV.
def save_dataframe(df_obj, path_base):
    try:
        df_obj.to_parquet(path_base + ".parquet", index=True)
        print("Saved:", path_base + ".parquet")
    except Exception as e:
        # fallback
        df_obj.to_csv(path_base + ".csv", index=True)
        print("parquet failed, saved CSV:", path_base + ".csv", " (error:", e, ")")

save_dataframe(train_df, os.path.join(OUT_DIR, "train"))
save_dataframe(test_df, os.path.join(OUT_DIR, "test"))
save_dataframe(results_df, os.path.join(OUT_DIR, "results_template"))


Saved: ../data/models\train.parquet
Saved: ../data/models\test.parquet
Saved: ../data/models\results_template.parquet


In [49]:
# --- 9) Sauvegarder préprocesseur et métadonnées ---
joblib.dump(preprocessor, os.path.join(ARTIFACTS_DIR, "preprocessor.joblib"))
print("Saved preprocessor to artifacts/preprocessor.joblib")

metadata = {
    "date_utc": datetime.datetime.utcnow().isoformat(),
    "random_state": RANDOM_STATE,
    "test_size": TEST_SIZE,
    "n_rows_total": df.shape[0],
    "n_features_raw": X.shape[1],
    "n_features_processed": len(feature_names),
    "train_shape": list(train_df.shape),
    "test_shape": list(test_df.shape),
    "train_churn_distribution": y_train.value_counts().to_dict(),
    "test_churn_distribution": y_test.value_counts().to_dict(),
    "feature_names": feature_names
}
with open(os.path.join(ARTIFACTS_DIR, "metadata.json"), "w") as f:
    json.dump(metadata, f, indent=2)
print("Saved metadata.json")


Saved preprocessor to artifacts/preprocessor.joblib
Saved metadata.json


C:\Users\User\AppData\Local\Temp\ipykernel_12940\225162100.py:6: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "date_utc": datetime.datetime.utcnow().isoformat(),


In [50]:
# Fin
print("Préparation terminée. Fichiers créés dans:", OUT_DIR, "et artefacts dans:", ARTIFACTS_DIR)

Préparation terminée. Fichiers créés dans: ../data/models et artefacts dans: ../data/models/artifacts
